In [0]:
# Databricks notebook source
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql import DataFrame
from datetime import datetime

print("🟤 INITIALIZING PROD BRONZE PIPELINE: BATCH PROCESSING")

# ==============================================================================
# 1. CONFIGURATION & WIDGETS
# ==============================================================================
# Explicitly define the environment physics
STORAGE_ACCOUNT = "stpraxaslakehouse"
RAW_CONTAINER = "bronze" 
DELTA_CONTAINER = "bronze"

# Unity Catalog Definitions (NO HIVE ALLOWED)
CATALOG_NAME = "prx" # Update this if your Unity Catalog has a different name
SCHEMA_NAME = "prx_bronze"

# Setup Dynamic Date Parameter for Orchestrator (ADF/Databricks Workflows)
dbutils.widgets.text("load_date", datetime.now().strftime("%Y-%m-%d"))
pipeline_date = dbutils.widgets.get("load_date")
year, month, day = pipeline_date.split("-")

print(f"📅 Target Execution Date: {pipeline_date}")

# Ensure Unity Catalog Schema exists securely
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")


# ==============================================================================
# 2. CORE INGESTION FRAMEWORK
# ==============================================================================


from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql import DataFrame

def load_to_bronze_delta(entity_name: str, exec_date: str, yr: str, mo: str, dy: str) -> None:
    # Construct exact ADLS paths
    raw_input_path = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/raw/praxas/{entity_name}/{yr}/{mo}/{dy}/*.json"
    delta_output_path = f"abfss://{DELTA_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/delta/praxas/{entity_name}"
    
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_{entity_name}"
    print(f"\n⏳ Processing Entity: [{entity_name.upper()}]")
    
    try:
        # Step 1: Read Raw JSON Data
        df_raw: DataFrame = spark.read \
            .option("inferSchema", "true") \
            .json(raw_input_path)

        print(df_raw.printSchema())

        
        financial_keywords = ["amount", "discount", "price", "vat", "cost", "tax", "qty", "quantity", "margin", "exrt", "fc", "dc"]
        string_keywords = ["id", "number", "code", "document", "zip", "phone", "fax", "account"]
        
        for col_name, dtype in df_raw.dtypes:
            col_lower = col_name.lower()
            
            # Rule 1: Fix Financial Math Columns (Force to Double)
            if any(kw in col_lower for kw in financial_keywords):
                if dtype in ('bigint', 'int', 'long'):
                    print(f"   🧬 Upgrading Math Column: {col_name} (from {dtype} to double)")
                    df_raw = df_raw.withColumn(col_name, col(col_name).cast("double"))
                    
            # Rule 2: Fix Identifiers & Codes (Force to String)
            elif any(kw in col_lower for kw in string_keywords):
                if dtype != 'string':
                    print(f"   🛡️ Shielding Identifier Column: {col_name} (from {dtype} to string)")
                    df_raw = df_raw.withColumn(col_name, col(col_name).cast("string"))

        # Step 2: Inject Enterprise Audit Columns
        df_audited: DataFrame = df_raw \
            .withColumn("sys_created_dt", current_timestamp()) \
            .withColumn("sys_created_by", lit("dbx_bronze_pipeline")) \
            .withColumn("source_file_date", lit(exec_date)) \
            .withColumn("source_file_path", col("_metadata.file_path")) 

        # Step 3: Write to Delta Lake
        df_audited.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .save(delta_output_path)

        # Step 4: Register to Unity Catalog
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {full_table_name}
            USING DELTA
            LOCATION '{delta_output_path}'
        """)

        print(f"✅ Success: {entity_name} safely landed in {full_table_name}")

    except Exception as e:
        print(f"❌ CRITICAL FAILURE: Could not process entity '{entity_name}'.")
        print(f"❌ Error Trace: {str(e)}")
        raise e


# ==============================================================================
# 3. PIPELINE EXECUTION
# ==============================================================================
# Loop through the entities to keep the code DRY
entities_to_process = [
    'addresses', 
    'salesorderheader', 
    'salesorderlines', 
    'warehouse', 
    'account'
]

for entity in entities_to_process:
    load_to_bronze_delta(entity, pipeline_date, year, month, day)

print("\n🎉 PROD BRONZE PIPELINE COMPLETED SUCCESSFULLY!")

# COMMAND ----------

# MAGIC %sql 
# MAGIC -- Verification Query using pure Unity Catalog syntax
# MAGIC SHOW TABLES IN main.praxas_bronze;